# 🤖 RNN Model — Waste Text Classification

This notebook builds a **Simple RNN model** for classifying waste items based on the cleaned text descriptions produced by the preprocessing notebook.

**Pipeline:**
1. Load Processed Data
2. Tokenization & Padding
3. Build RNN Model
4. Train the Model
5. Evaluate the Model
6. Save the Model

## 0. Install & Import Libraries

In [ ]:
import subprocess, sys

def install(pkg):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg, '-q'])

for pkg in ['tensorflow', 'scikit-learn', 'pandas', 'numpy', 'matplotlib', 'seaborn']:
    install(pkg)

print('✅ All packages installed.')

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    Embedding, SimpleRNN, Dense, Dropout, Bidirectional
)
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from tensorflow.keras.utils import to_categorical

print(f'✅ TensorFlow version: {tf.__version__}')
print(f'✅ All libraries imported.')

## 1. Load Processed Data

> **Note:** Make sure you've run the preprocessing notebook first so the `processed/` folder exists.

In [ ]:
# ── Load the splits produced by the preprocessing notebook ──
train_df = pd.read_csv('processed/train.csv')
val_df   = pd.read_csv('processed/val.csv')
test_df  = pd.read_csv('processed/test.csv')

print(f'Train : {len(train_df):,} rows')
print(f'Val   : {len(val_df):,} rows')
print(f'Test  : {len(test_df):,} rows')

# ── Use the cleaned text column from preprocessing ──
TEXT_COL  = 'text_clean'   # cleaned column from the preprocessing notebook
LABEL_COL = 'label'

X_train_raw = train_df[TEXT_COL].astype(str).tolist()
X_val_raw   = val_df[TEXT_COL].astype(str).tolist()
X_test_raw  = test_df[TEXT_COL].astype(str).tolist()

y_train_raw = train_df[LABEL_COL].tolist()
y_val_raw   = val_df[LABEL_COL].tolist()
y_test_raw  = test_df[LABEL_COL].tolist()

print(f'\nSample text  : {X_train_raw[0]}')
print(f'Sample label : {y_train_raw[0]}')

## 2. Encode Labels

In [ ]:
le = LabelEncoder()
le.fit(y_train_raw)

y_train = le.transform(y_train_raw)
y_val   = le.transform(y_val_raw)
y_test  = le.transform(y_test_raw)

NUM_CLASSES = len(le.classes_)
print(f'Number of classes: {NUM_CLASSES}')
print(f'Classes          : {list(le.classes_)}')

# Convert to one-hot for categorical_crossentropy
y_train_cat = to_categorical(y_train, NUM_CLASSES)
y_val_cat   = to_categorical(y_val,   NUM_CLASSES)
y_test_cat  = to_categorical(y_test,  NUM_CLASSES)

## 3. Tokenization & Padding

In [ ]:
# ── Hyperparameters ──
VOCAB_SIZE  = 10_000   # keep the top-N most frequent words
MAX_LEN     = 50       # max sequence length (pad/truncate)
OOV_TOKEN   = '<OOV>'  # out-of-vocabulary token

# ── Fit tokenizer on TRAINING data only ──
tokenizer = Tokenizer(num_words=VOCAB_SIZE, oov_token=OOV_TOKEN)
tokenizer.fit_on_texts(X_train_raw)

print(f'Vocabulary size (actual): {len(tokenizer.word_index):,}')
print(f'Vocabulary size (capped) : {VOCAB_SIZE:,}')

# ── Text → sequences → padded arrays ──
def encode(texts):
    seqs = tokenizer.texts_to_sequences(texts)
    return pad_sequences(seqs, maxlen=MAX_LEN, padding='post', truncating='post')

X_train = encode(X_train_raw)
X_val   = encode(X_val_raw)
X_test  = encode(X_test_raw)

print(f'\nX_train shape: {X_train.shape}')
print(f'X_val   shape: {X_val.shape}')
print(f'X_test  shape: {X_test.shape}')

## 4. Build the RNN Model

In [ ]:
# ── Model Hyperparameters ──
EMBED_DIM   = 64    # embedding dimension
RNN_UNITS   = 128   # hidden units in the RNN layer
DROPOUT     = 0.4   # dropout rate

def build_rnn(vocab_size, embed_dim, rnn_units, num_classes, dropout, max_len):
    model = Sequential([
        # ── 1. Embedding layer: integer token → dense vector ──
        Embedding(
            input_dim=vocab_size,
            output_dim=embed_dim,
            input_length=max_len,
            name='embedding'
        ),

        # ── 2. Bidirectional SimpleRNN ──
        #    Wrapping in Bidirectional lets the model read the sequence
        #    both left-to-right and right-to-left, which usually helps
        #    short text classification tasks.
        Bidirectional(
            SimpleRNN(rnn_units, return_sequences=False),
            name='bi_rnn'
        ),

        # ── 3. Regularisation ──
        Dropout(dropout, name='dropout'),

        # ── 4. Dense hidden layer ──
        Dense(64, activation='relu', name='dense_hidden'),
        Dropout(dropout / 2, name='dropout_2'),

        # ── 5. Output layer ──
        Dense(num_classes, activation='softmax', name='output')
    ], name='WasteRNN')

    return model


model = build_rnn(
    vocab_size  = VOCAB_SIZE,
    embed_dim   = EMBED_DIM,
    rnn_units   = RNN_UNITS,
    num_classes = NUM_CLASSES,
    dropout     = DROPOUT,
    max_len     = MAX_LEN
)

model.summary()

## 5. Compile the Model

In [ ]:
model.compile(
    optimizer = tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss      = 'categorical_crossentropy',
    metrics   = ['accuracy']
)

print('✅ Model compiled.')

## 6. Train the Model

In [ ]:
EPOCHS     = 30
BATCH_SIZE = 32

callbacks = [
    EarlyStopping(
        monitor   = 'val_loss',
        patience  = 5,
        restore_best_weights = True,
        verbose   = 1
    ),
    ModelCheckpoint(
        filepath  = 'best_rnn_model.keras',
        monitor   = 'val_accuracy',
        save_best_only = True,
        verbose   = 1
    )
]

history = model.fit(
    X_train, y_train_cat,
    validation_data = (X_val, y_val_cat),
    epochs          = EPOCHS,
    batch_size      = BATCH_SIZE,
    callbacks       = callbacks,
    verbose         = 1
)

## 7. Plot Training History

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ── Accuracy ──
axes[0].plot(history.history['accuracy'],     label='Train Accuracy', color='steelblue')
axes[0].plot(history.history['val_accuracy'], label='Val Accuracy',   color='orange')
axes[0].set_title('Accuracy over Epochs', fontweight='bold')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Accuracy')
axes[0].legend()
axes[0].grid(True)

# ── Loss ──
axes[1].plot(history.history['loss'],     label='Train Loss', color='steelblue')
axes[1].plot(history.history['val_loss'], label='Val Loss',   color='orange')
axes[1].set_title('Loss over Epochs', fontweight='bold')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss')
axes[1].legend()
axes[1].grid(True)

plt.tight_layout()
plt.show()

## 8. Evaluate on Test Set

In [ ]:
test_loss, test_acc = model.evaluate(X_test, y_test_cat, verbose=0)
print(f'Test Loss     : {test_loss:.4f}')
print(f'Test Accuracy : {test_acc*100:.2f}%')

In [ ]:
# ── Per-class classification report ──
y_pred_probs = model.predict(X_test, verbose=0)
y_pred       = np.argmax(y_pred_probs, axis=1)

print('Classification Report:')
print('=' * 55)
print(classification_report(
    y_test,
    y_pred,
    target_names = le.classes_
))

In [ ]:
# ── Confusion Matrix ──
cm = confusion_matrix(y_test, y_pred)

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(
    cm,
    annot      = True,
    fmt        = 'd',
    cmap       = 'Blues',
    xticklabels = le.classes_,
    yticklabels = le.classes_,
    ax         = ax
)
ax.set_title('Confusion Matrix — Test Set', fontweight='bold', fontsize=13)
ax.set_xlabel('Predicted Label')
ax.set_ylabel('True Label')
plt.tight_layout()
plt.show()

## 9. Inference — Predict a New Sample

In [ ]:
def predict_waste(text: str) -> str:
    """
    Predict the waste category for a single raw text string.
    The same cleaning used at training time is expected to have already
    been applied, OR you can apply the clean_text() function from the
    preprocessing notebook here before encoding.
    """
    seq     = tokenizer.texts_to_sequences([text])
    padded  = pad_sequences(seq, maxlen=MAX_LEN, padding='post', truncating='post')
    probs   = model.predict(padded, verbose=0)[0]
    idx     = np.argmax(probs)
    label   = le.inverse_transform([idx])[0]
    conf    = probs[idx] * 100
    return f'Predicted: {label}  (confidence: {conf:.1f}%)'


# ── Quick test ──
sample_texts = [
    'plastic bottle crushed recyclable',
    'food leftover organic kitchen waste',
    'broken glass sharp dangerous'
]

for t in sample_texts:
    print(f'Input   : {t}')
    print(f'Output  : {predict_waste(t)}')
    print('-' * 45)

## 10. Save the Model

In [ ]:
import os, pickle

os.makedirs('saved_model', exist_ok=True)

# ── Save Keras model ──
model.save('saved_model/rnn_waste_model.keras')

# ── Save tokenizer & label encoder (needed for inference) ──
with open('saved_model/tokenizer.pkl', 'wb') as f:
    pickle.dump(tokenizer, f)

with open('saved_model/label_encoder.pkl', 'wb') as f:
    pickle.dump(le, f)

print('✅ Model saved to ./saved_model/')
print('   ├── rnn_waste_model.keras')
print('   ├── tokenizer.pkl')
print('   └── label_encoder.pkl')